<a href="https://colab.research.google.com/github/sathvika-source/R-Sathvika-DAA-practical01-8939/blob/main/DAA_Practical_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
def solve_tsp_dynamic_programming(distance_matrix):
    n = len(distance_matrix)
    # memo table stores (subset_mask, current_vertex) -> (min_distance, parent_vertex)
    memo = {}

    # Base cases: distance from starting node (0) to any node i with subset containing only {0, i}
    for i in range(1, n):
        memo[(1 << 0) | (1 << i), i] = (distance_matrix[0][i], 0)

    # Iterate through subset sizes
    for subset_size in range(3, n + 1):
        for subset in range(1 << n):
            if not (subset & 1): # Subset must contain starting node 0
                continue
            if bin(subset).count('1') != subset_size:
                continue

            for next_node in range(1, n):
                if not (subset & (1 << next_node)):
                    continue

                # Find the minimum path to next_node using previous subproblems
                prev_subset = subset ^ (1 << next_node)
                min_dist = float('inf')
                best_parent = None

                for last_node in range(1, n):
                    if last_node == next_node or not (prev_subset & (1 << last_node)):
                        continue

                    cost = memo[(prev_subset, last_node)][0] + distance_matrix[last_node][next_node]
                    if cost < min_dist:
                        min_dist = cost
                        best_parent = last_node

                memo[(subset, next_node)] = (min_dist, best_parent)

    # Find the complete tour returning to start node 0
    full_subset = (1 << n) - 1
    min_total_dist = float('inf')
    last_node_in_tour = None

    for last_node in range(1, n):
        cost = memo[(full_subset, last_node)][0] + distance_matrix[last_node][0]
        if cost < min_total_dist:
            min_total_dist = cost
            last_node_in_tour = last_node

    # Reconstruct the optimal path
    path = [0]
    current_subset = full_subset
    curr_node = last_node_in_tour

    while curr_node != 0:
        path.append(curr_node)
        prev_subset = current_subset ^ (1 << curr_node)
        curr_node = memo[(current_subset, curr_node)][1]
        current_subset = prev_subset

    path.append(0)
    path.reverse() # Reverse path to trace from start to finish

    return path, min_total_dist

# --- Example Distance Matrix (for 4 cities) ---
# Matrix representations where matrix[i][j] is the distance from i to j
example_distances = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

optimal_path, min_distance = solve_tsp_dynamic_programming(example_distances)
print("Optimal TSP Tour Path:", optimal_path)
print("Minimum Tour Distance:", min_distance)

Optimal TSP Tour Path: [0, 2, 3, 1, 0]
Minimum Tour Distance: 80
